In [1]:
# Step 01: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import warnings
warnings.filterwarnings('ignore')

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 3.2.0


In [ ]:
# Step 02: Loading Engineered Features

In [3]:
df_feat = pd.read_csv('../data/processed/features_engineered.csv', parse_dates=['date'])
print(f"Shape: {df_feat.shape}")
print(f"Columns: {df_feat.columns.tolist()}")

Shape: (730500, 30)
Columns: ['date', 'store', 'item', 'sales', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_90', 'lag_180', 'lag_365', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30', 'rolling_mean_90', 'rolling_std_90', 'ewm_7', 'days_since_start']


In [ ]:
# Step 03: Define the features and split

In [5]:
FEATURE_COLS = [
    'store', 'item',
    'year', 'month', 'day', 'day_of_week', 'week_of_year',
    'quarter', 'is_weekend', 'is_month_start', 'is_month_end',
    'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_90', 'lag_180', 'lag_365',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_mean_90',
    'rolling_std_7', 'rolling_std_14', 'rolling_std_30', 'rolling_std_90',
    'ewm_7', 'days_since_start'
]
TARGET = 'sales'

SPLIT_DATE = '2017-10-01'
train_df = df_feat[df_feat['date'] < SPLIT_DATE]
test_df  = df_feat[df_feat['date'] >= SPLIT_DATE]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")
print(f"Number of features: {len(FEATURE_COLS)}")

Training samples: 684,500
Test samples: 46,000
Number of features: 28


In [ ]:
# Step 04: Training XGBoost

In [7]:
model_xgb = xgb.XGBRegressor(
    n_estimators = 1000,
    learning_rate = 0.05,
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 1,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    objective = 'reg:squarederror',
    early_stopping_rounds = 50,
    eval_metric = 'rmse',
    random_state = 42,
    n_jobs = -1
)

print("Training XGBoost")
model_xgb.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
    verbose = 200     # Printing the progress every 200 trees
)
print("\nTraining complete!")
print(f"Best iteration: {model_xgb.best_iteration}")

Training XGBoost
[0]	validation_0-rmse:27.21080
[200]	validation_0-rmse:7.75160
[400]	validation_0-rmse:7.67297
[600]	validation_0-rmse:7.64531
[787]	validation_0-rmse:7.63750

Training complete!
Best iteration: 737


In [ ]:
# Step 05: Evaluation

In [11]:
y_pred = model_xgb.predict(X_test)
y_pred = np.maximum(y_pred, 0)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = (abs(y_test - y_pred) / (y_test + 1e-5)).mean() * 100

print("XGBOOST MODEL RESULTS — All Stores & Items")
print(f"MAE: {mae:.2f} units")
print(f"RMSE: {rmse:.2f} units")
print(f"MAPE: {mape:.2f}%")

XGBOOST MODEL RESULTS — All Stores & Items
MAE: 5.90 units
RMSE: 7.64 units
MAPE: 12.97%


In [ ]:
# Step 06: Feature importance chart

In [ ]:
feat_imp = pd.Series(model_xgb.feature_importances_, index=FEATURE_COLS)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 7))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('XGBoost Feature Importance — Top 15 Most Useful Features', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../results/xgb_feature_importance.png', dpi=150)
plt.show()

print(f"\nMost important feature: {feat_imp.index[-1]}")
print("This feature tells us the most about future sales.")

In [ ]:
# Step 07: Now we save the model

In [12]:
with open('../models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(model_xgb, f)
print("Model saved to models/xgboost_model.pkl")

Model saved to models/xgboost_model.pkl
